<a href="https://colab.research.google.com/github/Vidit-01/pedestrian_intent_prediction/blob/yash/experiments/paper1_vlfm_jaad/notebooks/01_jaad_annotation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import os
import subprocess

REPO_URL = "https://github.com/Vidit-01/pedestrian_intent_prediction.git"
BRANCH = "yash"
REPO_DIR = "/content/pedestrian_intent_prediction"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch",
         REPO_URL, REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        check=True
    )
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "origin", BRANCH],
        check=True
    )

os.chdir(REPO_DIR)
print("Repository:", os.getcwd())
print("Branch:", subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())
print("Latest commit:", subprocess.check_output(
    ["git", "log", "-1", "--oneline"], text=True
).strip())


Repository: /content/pedestrian_intent_prediction
Branch: yash
Latest commit: 96a6460 data exploration


In [3]:

from pathlib import Path
import sys
import os
import json

REPO_ROOT = Path("/content/pedestrian_intent_prediction")
JAAD_ROOT = Path("/content/JAAD")
EXPERIMENT_ROOT = (
    REPO_ROOT / "experiments" / "paper1_vlfm_jaad"
)

print("Python:", sys.version.split()[0])
print("Repository exists:", REPO_ROOT.exists())
print("JAAD directory exists:", JAAD_ROOT.exists())
print("Experiment directory exists:", EXPERIMENT_ROOT.exists())

assert REPO_ROOT.exists(), "Repository not found. Check your Colab paths."


Python: 3.13.16
Repository exists: True
JAAD directory exists: False
Experiment directory exists: True


In [4]:
from pathlib import Path
import subprocess

JAAD_ROOT = Path("/content/JAAD")

if not (JAAD_ROOT / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth", "1",
            "https://github.com/ykotseruba/JAAD.git",
            str(JAAD_ROOT)
        ],
        check=True
    )
else:
    print("JAAD repository already cloned.")

print("JAAD directory exists:", JAAD_ROOT.exists())
print("JAAD contents:")
for item in list(JAAD_ROOT.iterdir())[:15]:
    print(item.name)

JAAD directory exists: True
JAAD contents:
annotations
annotations_appearance
annotations_attributes
LICENSE
behavior.png
annotations_vehicle
README.md
annotations_traffic
download_clips.sh
split_ids
.git
jaad_data.py
split_clips_to_frames.sh


In [5]:
from pathlib import Path

JAAD_ROOT = Path("/content/JAAD")

annotation_dirs = [
    "annotations",
    "annotations_appearance",
    "annotations_attributes",
    "annotations_vehicle",
    "annotations_traffic",
    "split_ids",
]

for dirname in annotation_dirs:
    folder = JAAD_ROOT / dirname
    print(f"\n{'=' * 55}")
    print(f"{dirname} | exists: {folder.exists()}")

    if folder.exists():
        items = sorted(folder.iterdir())
        print(f"Number of entries: {len(items)}")

        for item in items[:8]:
            if item.is_file():
                print(f"  {item.name} ({item.stat().st_size:,} bytes)")
            else:
                print(f"  {item.name}/")


annotations | exists: True
Number of entries: 346
  video_0001.xml (442,767 bytes)
  video_0002.xml (186,979 bytes)
  video_0003.xml (168,655 bytes)
  video_0004.xml (105,395 bytes)
  video_0005.xml (578,775 bytes)
  video_0006.xml (1,337,077 bytes)
  video_0007.xml (93,457 bytes)
  video_0008.xml (84,761 bytes)

annotations_appearance | exists: True
Number of entries: 346
  video_0001_appearance.xml (435,203 bytes)
  video_0002_appearance.xml (141,451 bytes)
  video_0003_appearance.xml (199,917 bytes)
  video_0004_appearance.xml (119,127 bytes)
  video_0005_appearance.xml (485,190 bytes)
  video_0006_appearance.xml (1,865,922 bytes)
  video_0007_appearance.xml (117,526 bytes)
  video_0008_appearance.xml (86,058 bytes)

annotations_attributes | exists: True
Number of entries: 346
  video_0001_attributes.xml (547 bytes)
  video_0002_attributes.xml (535 bytes)
  video_0003_attributes.xml (285 bytes)
  video_0004_attributes.xml (288 bytes)
  video_0005_attributes.xml (1,564 bytes)
  vide

In [6]:
import ast

parser_path = JAAD_ROOT / "jaad_data.py"
source = parser_path.read_text(encoding="utf-8")
tree = ast.parse(source)

print("Classes and methods defined in jaad_data.py:\n")

for node in tree.body:
    if isinstance(node, ast.ClassDef):
        print(f"\nCLASS: {node.name}")

        for method in node.body:
            if isinstance(method, (ast.FunctionDef, ast.AsyncFunctionDef)):
                print(f"  {method.name}()")

Classes and methods defined in jaad_data.py:


CLASS: JAAD
  __init__()
  cache_path()
  _get_default_path()
  _get_video_ids_split()
  _get_video_ids()
  _get_image_path()
  update_progress()
  _print_dict()
  _squarify()
  extract_and_save_images()
  _map_text_to_scalar()
  _map_scalar_to_text()
  _get_annotations()
  _get_ped_attributes()
  _get_ped_appearance()
  _get_traffic_attributes()
  _get_vehicle_attributes()
  generate_database()
  get_data_stats()
  balance_samples_count()
  _get_pedestrian_ids()
  _get_random_pedestrian_ids()
  _get_kfold_pedestrian_ids()
  get_detection_data()
  _get_data_frcnn()
  _generate_csv_data_retinanet()
  _generate_csv_data_yolo3()
  _generate_csv_data_ssd()
  _get_data_ids()
  _height_check()
  _get_center()
  generate_data_trajectory_sequence()
  _get_trajectories()
  _get_crossing()
  _get_intention()


In [7]:
import xml.etree.ElementTree as ET
from pathlib import Path

JAAD_ROOT = Path("/content/JAAD")

xml_path = JAAD_ROOT / "annotations" / "video_0001.xml"
tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)
print("Root attributes:", root.attrib)

print("\nTop-level elements:")
for child in list(root)[:10]:
    print(
        "Tag:", child.tag,
        "| Attributes:", child.attrib,
        "| Children:", len(child)
    )

print("\nFirst annotation subtree:")
first_child = next(iter(root), None)

if first_child is not None:
    print(ET.tostring(first_child, encoding="unicode")[:4000])

Root tag: annotations
Root attributes: {}

Top-level elements:
Tag: version | Attributes: {} | Children: 0
Tag: meta | Attributes: {} | Children: 2
Tag: track | Attributes: {'label': 'pedestrian'} | Children: 569
Tag: track | Attributes: {'label': 'ped'} | Children: 221
Tag: track | Attributes: {'label': 'pedestrian'} | Children: 70
Tag: track | Attributes: {'label': 'ped'} | Children: 217

First annotation subtree:
<version>1.1</version>


In [8]:
import inspect
import sys

sys.path.insert(0, str(JAAD_ROOT))

from jaad_data import JAAD

for method_name in [
    "_get_annotations",
    "_get_ped_attributes",
    "_get_crossing",
    "_get_intention",
    "generate_data_trajectory_sequence",
]:
    method = getattr(JAAD, method_name)

    print(f"\n{'=' * 70}")
    print(f"METHOD: {method_name}")
    print(inspect.getsource(method))


METHOD: _get_annotations
    def _get_annotations(self, vid):
        """
        Generates a dictinary of annotations by parsing the video XML file
        :param vid: The id of video to parse
        :return: A dictionary of annotations
        """
        path_to_file = join(self._annotation_path, vid + '.xml')
        tree = ET.parse(path_to_file)
        ped_annt = 'ped_annotations'

        annotations = {}
        annotations['num_frames'] = int(tree.find("./meta/task/size").text)
        annotations['width'] = int(tree.find("./meta/task/original_size/width").text)
        annotations['height'] = int(tree.find("./meta/task/original_size/height").text)
        annotations[ped_annt] = {}

        ped_tracks = tree.findall("./track")

        for t in ped_tracks:
            boxes = t.findall('./box')
            new_id = boxes[0].find('./attribute[@name=\"id\"]').text
            old_id = boxes[0].find('./attribute[@name=\"old_id\"]').text
            annotations[ped_annt][new_id

In [9]:
import xml.etree.ElementTree as ET
from pathlib import Path

JAAD_ROOT = Path("/content/JAAD")
xml_path = JAAD_ROOT / "annotations" / "video_0001.xml"

root = ET.parse(xml_path).getroot()

for track in root.findall("./track")[:4]:
    boxes = track.findall("./box")

    print("\nTrack label:", track.get("label"))
    print("Number of boxes:", len(boxes))

    if boxes:
        first_box = boxes[0]

        print("First box attributes:", first_box.attrib)

        print("Attributes inside first box:")
        for attr in first_box.findall("./attribute"):
            print(f"  {attr.get('name')}: {attr.text}")


Track label: pedestrian
Number of boxes: 569
First box attributes: {'frame': '0', 'keyframe': '1', 'occluded': '0', 'outside': '0', 'xbr': '533.0', 'xtl': '465.0', 'ybr': '848.0', 'ytl': '730.0'}
Attributes inside first box:
  id: 0_1_3b
  old_id: pedestrian1
  look: not-looking
  reaction: __undefined__
  action: standing
  cross: not-crossing
  hand_gesture: __undefined__
  occlusion: none
  nod: __undefined__

Track label: ped
Number of boxes: 221
First box attributes: {'frame': '379', 'keyframe': '1', 'occluded': '1', 'outside': '0', 'xbr': '186.0', 'xtl': '150.0', 'ybr': '796.0', 'ytl': '684.0'}
Attributes inside first box:
  id: 0_1_2
  old_id: ped2
  occlusion: full

Track label: pedestrian
Number of boxes: 70
First box attributes: {'frame': '0', 'keyframe': '1', 'occluded': '0', 'outside': '0', 'xbr': '1486.0', 'xtl': '1398.0', 'ybr': '892.0', 'ytl': '654.0'}
Attributes inside first box:
  id: 0_1_2b
  old_id: pedestrian2
  look: not-looking
  reaction: __undefined__
  action:

In [10]:
attributes_path = (
    JAAD_ROOT / "annotations_attributes" / "video_0001_attributes.xml"
)

print(attributes_path.read_text(encoding="utf-8")[:5000])

<ped_attributes><pedestrian age="adult" crossing="-1" crossing_point="-1" decision_point="26" designated="ND" gender="female" group_size="1" id="0_1_2b" intersection="no" motion_direction="LONG" num_lanes="2" old_id="pedestrian2" signalized="n/a" traffic_direction="TW" /><pedestrian age="senior" crossing="-1" crossing_point="-1" decision_point="363" designated="ND" gender="female" group_size="1" id="0_1_3b" intersection="no" motion_direction="LONG" num_lanes="2" old_id="pedestrian1" signalized="n/a" traffic_direction="TW" /></ped_attributes>


In [11]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

JAAD_ROOT = Path("/content/JAAD")

track_label_counts = Counter()
crossing_counts = Counter()
missing_attribute_files = []
track_count = 0
attribute_count = 0
matched_count = 0

for xml_path in sorted((JAAD_ROOT / "annotations").glob("*.xml")):
    video_id = xml_path.stem
    attr_path = (
        JAAD_ROOT / "annotations_attributes"
        / f"{video_id}_attributes.xml"
    )

    if not attr_path.exists():
        missing_attribute_files.append(video_id)
        continue

    main_root = ET.parse(xml_path).getroot()
    attr_root = ET.parse(attr_path).getroot()

    track_ids = set()

    for track in main_root.findall("./track"):
        track_label_counts[track.get("label")] += 1
        track_count += 1

        first_box = track.find("./box")
        if first_box is not None:
            id_element = first_box.find("./attribute[@name='id']")
            if id_element is not None and id_element.text:
                track_ids.add(id_element.text)

    for pedestrian in attr_root.findall("./pedestrian"):
        attribute_count += 1
        crossing_counts[pedestrian.get("crossing", "MISSING")] += 1

        if pedestrian.get("id") in track_ids:
            matched_count += 1

print("Track counts by XML label:", dict(track_label_counts))
print("Crossing attribute values:", dict(crossing_counts))
print("Total tracks:", track_count)
print("Total pedestrian attribute records:", attribute_count)
print("Attribute records matching a main XML track:", matched_count)
print("Videos missing attribute files:", missing_attribute_files)

Track counts by XML label: {'pedestrian': 686, 'ped': 2017, 'people': 83}
Crossing attribute values: {'-1': 100, '1': 495, '0': 91}
Total tracks: 2786
Total pedestrian attribute records: 686
Attribute records matching a main XML track: 686
Videos missing attribute files: []


In [12]:
from pathlib import Path

JAAD_ROOT = Path("/content/JAAD")

readme = JAAD_ROOT / "README.md"
print("===== JAAD README =====")
print(readme.read_text(encoding="utf-8")[:12000])

print("\n===== LABEL MAPPING METHODS =====")

import inspect
import sys

sys.path.insert(0, str(JAAD_ROOT))
from jaad_data import JAAD

for name in ["_map_text_to_scalar", "_map_scalar_to_text"]:
    print(f"\n--- {name} ---")
    print(inspect.getsource(getattr(JAAD, name)))

===== JAAD README =====
# JAAD 2.0: Annotations and python interface
<p align="center">
<img src="behavior.png" alt="jaad_samples" align="middle" width="600"/>
</p>
<br/><br/>

This repository contains new annotations for the Joint Attention in Autonomous Driving ([JAAD](http://data.nvision2.eecs.yorku.ca/JAAD_dataset/)) dataset. The annotations are in XML format and can be used with a newly introduced python interface. The original annotations can be found [here](https://github.com/ykotseruba/JAAD_pedestrian/tree/master).  

**Download video clips:** [YorkU server](http://data.nvision2.eecs.yorku.ca/JAAD_dataset/data/JAAD_clips.zip) [Google Drive](https://drive.google.com/file/d/1HCFLBO9fJutCKG11FtjKfdLvME6Qe_5L/view?usp=drive_link)

The total download size is approx. 3.1GB.

### Table of contents
* [Annotations](#annotations)
* [Video clips](#clips)
* [Interface](#interface)
	* [Dependencies](#dependencies)
	* [Extracting images](#extracting)
	* [Using the interface](#usage)
		* [Par

In [13]:
from pathlib import Path

JAAD_ROOT = Path("/content/JAAD")
SPLIT_ROOT = JAAD_ROOT / "split_ids"

for subset in ["default", "high_visibility", "all_videos"]:
    folder = SPLIT_ROOT / subset

    print(f"\n{'=' * 55}")
    print("Subset:", subset)
    print("Exists:", folder.exists())

    if folder.exists():
        for path in sorted(folder.iterdir()):
            print(f"\nFile: {path.name}")

            if path.is_file():
                content = path.read_text(encoding="utf-8")
                print(content[:1200])


Subset: default
Exists: True

File: test.txt
video_0005
video_0015
video_0016
video_0017
video_0028
video_0036
video_0042
video_0043
video_0045
video_0046
video_0048
video_0053
video_0055
video_0058
video_0059
video_0071
video_0075
video_0076
video_0084
video_0087
video_0090
video_0092
video_0093
video_0096
video_0097
video_0100
video_0101
video_0103
video_0104
video_0105
video_0106
video_0107
video_0110
video_0113
video_0115
video_0116
video_0117
video_0118
video_0124
video_0125
video_0128
video_0135
video_0141
video_0144
video_0148
video_0150
video_0151
video_0152
video_0153
video_0155
video_0162
video_0163
video_0164
video_0165
video_0173
video_0177
video_0178
video_0179
video_0183
video_0187
video_0197
video_0201
video_0203
video_0206
video_0211
video_0212
video_0213
video_0216
video_0221
video_0222
video_0223
video_0224
video_0230
video_0234
video_0238
video_0239
video_0243
video_0244
video_0245
video_0251
video_0253
video_0265
video_0267
video_0270
video_0271
video_0277
video_02

In [14]:

from pathlib import Path
import xml.etree.ElementTree as ET

JAAD_ROOT = Path("/content/JAAD")
SPLIT_ROOT = JAAD_ROOT / "split_ids"

def read_split(folder, split):
    path = folder / f"{split}.txt"
    return {
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    }

splits = {
    name: read_split(SPLIT_ROOT / "default", name)
    for name in ["train", "val", "test"]
}

# 1. Split sizes and overlaps
print("OFFICIAL DEFAULT SPLIT AUDIT")
for name, video_ids in splits.items():
    print(f"{name:>5}: {len(video_ids)} videos")

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = splits[a] & splits[b]
    print(f"{a} ∩ {b}: {len(overlap)} overlapping videos")

# 2. Check duplicates in the original text files
for name in ["train", "val", "test"]:
    lines = [
        line.strip()
        for line in (SPLIT_ROOT / "default" / f"{name}.txt")
        .read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    duplicates = sorted({x for x in lines if lines.count(x) > 1})
    print(f"{name} duplicate entries:", duplicates[:10] or "None")

# 3. Check annotation availability
annotation_ids = {
    p.stem for p in (JAAD_ROOT / "annotations").glob("*.xml")
}

all_split_ids = set().union(*splits.values())
print("\nTotal unique videos in splits:", len(all_split_ids))
print("Annotated videos:", len(annotation_ids))
print("Split IDs without annotation XML:",
      sorted(all_split_ids - annotation_ids))
print("Annotated videos absent from default splits:",
      sorted(annotation_ids - all_split_ids))

# 4. Confirm the subset's split files are distinct
for subset in ["default", "high_visibility", "all_videos"]:
    folder = SPLIT_ROOT / subset
    counts = {
        name: len(read_split(folder, name))
        for name in ["train", "val", "test"]
    }
    print(f"{subset}: {counts}")


OFFICIAL DEFAULT SPLIT AUDIT
train: 177 videos
  val: 29 videos
 test: 117 videos
train ∩ val: 0 overlapping videos
train ∩ test: 0 overlapping videos
val ∩ test: 0 overlapping videos
train duplicate entries: None
val duplicate entries: None
test duplicate entries: None

Total unique videos in splits: 323
Annotated videos: 346
Split IDs without annotation XML: []
Annotated videos absent from default splits: ['video_0002', 'video_0013', 'video_0022', 'video_0023', 'video_0024', 'video_0026', 'video_0029', 'video_0032', 'video_0033', 'video_0034', 'video_0041', 'video_0061', 'video_0062', 'video_0063', 'video_0064', 'video_0065', 'video_0066', 'video_0067', 'video_0068', 'video_0069', 'video_0070', 'video_0127', 'video_0346']
default: {'train': 177, 'val': 29, 'test': 117}
high_visibility: {'train': 182, 'val': 30, 'test': 121}
all_videos: {'train': 188, 'val': 32, 'test': 126}


In [15]:

from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

JAAD_ROOT = Path("/content/JAAD")
ANNOTATIONS = JAAD_ROOT / "annotations"
ATTRIBUTES = JAAD_ROOT / "annotations_attributes"

rows = []

for xml_path in sorted(ANNOTATIONS.glob("*.xml")):
    video_id = xml_path.stem
    root = ET.parse(xml_path).getroot()

    # Index track-level attributes by the track ID.
    attribute_path = ATTRIBUTES / f"{video_id}_attributes.xml"
    attr_by_id = {}

    if attribute_path.exists():
        attr_root = ET.parse(attribute_path).getroot()

        for element in attr_root.iter():
            if element.tag.lower() in {"pedestrian", "ped"}:
                track_id = element.attrib.get("id")
                if track_id:
                    attr_by_id[track_id] = element.attrib.copy()

    for track in root.findall(".//track"):
        label = track.attrib.get("label", "")
        boxes = track.findall("box")

        if not boxes:
            continue

        first_box = boxes[0]
        track_id = first_box.findtext("attribute[@name='id']")
        old_id = first_box.findtext("attribute[@name='old_id']")

        # Some JAAD versions store these attributes on the track itself.
        track_id = track_id or track.attrib.get("id")
        old_id = old_id or track.attrib.get("old_id")

        if not track_id:
            continue

        attrs = attr_by_id.get(track_id, {})

        def box_attribute(box, name):
            element = box.find(f"attribute[@name='{name}']")
            return element.text if element is not None else None

        rows.append({
            "video_id": video_id,
            "track_id": track_id,
            "old_id": old_id,
            "xml_label": label,
            "num_boxes": len(boxes),
            "first_frame": first_box.attrib.get("frame"),
            "last_frame": boxes[-1].attrib.get("frame"),
            "frame_cross_first": box_attribute(first_box, "cross"),
            "frame_action_first": box_attribute(first_box, "action"),
            "frame_look_first": box_attribute(first_box, "look"),
            "track_crossing": attrs.get("crossing"),
            "crossing_point": attrs.get("crossing_point"),
            "decision_point": attrs.get("decision_point"),
            "has_track_attributes": bool(attrs),
        })

tracks_df = pd.DataFrame(rows)

print("Total parsed tracks:", len(tracks_df))
print("\nTrack labels:")
print(tracks_df["xml_label"].value_counts(dropna=False))

print("\nTracks with track-level attribute records:")
print(tracks_df["has_track_attributes"].value_counts(dropna=False))

print("\nTrack-level crossing values:")
print(tracks_df["track_crossing"].value_counts(dropna=False))

print("\nFrame-level cross values on first box:")
print(tracks_df["frame_cross_first"].value_counts(dropna=False))

print("\nExample behaviour-annotated tracks:")
display(
    tracks_df[
        tracks_df["has_track_attributes"]
    ].head(10)
)


Total parsed tracks: 2786

Track labels:
xml_label
ped           2017
pedestrian     686
people          83
Name: count, dtype: int64

Tracks with track-level attribute records:
has_track_attributes
False    2100
True      686
Name: count, dtype: int64

Track-level crossing values:
track_crossing
None    2100
1        495
-1       100
0         91
Name: count, dtype: int64

Frame-level cross values on first box:
frame_cross_first
None            2100
not-crossing     464
crossing         222
Name: count, dtype: int64

Example behaviour-annotated tracks:


,video_id,track_id,old_id,xml_label,num_boxes,first_frame,last_frame,frame_cross_first,frame_action_first,frame_look_first,track_crossing,crossing_point,decision_point,has_track_attributes
0,video_0001,0_1_3b,pedestrian1,pedestrian,569,0,568,not-crossing,standing,not-looking,-1,-1,363,True
2,video_0001,0_1_2b,pedestrian2,pedestrian,70,0,69,not-crossing,walking,not-looking,-1,-1,26,True
4,video_0002,0_2_5b,pedestrian1,pedestrian,192,18,209,not-crossing,walking,not-looking,1,53,43,True
5,video_0002,0_2_6b,pedestrian2,pedestrian,158,52,209,not-crossing,walking,not-looking,1,81,73,True
8,video_0003,0_3_7b,pedestrian,pedestrian,175,0,174,crossing,walking,not-looking,1,-1,-1,True
10,video_0004,0_4_10b,pedestrian,pedestrian,119,0,118,not-crossing,walking,not-looking,0,0,33,True
11,video_0005,0_5_19b,pedestrian1,pedestrian,180,0,179,not-crossing,walking,not-looking,-1,0,80,True
12,video_0005,0_5_16b,pedestrian3,pedestrian,158,0,157,not-crossing,walking,looking,-1,-1,42,True
13,video_0005,0_5_14b,pedestrian2,pedestrian,181,0,180,not-crossing,walking,not-looking,-1,-1,42,True
14,video_0005,0_5_18b,pedestrian4,pedestrian,124,0,123,not-crossing,walking,not-looking,-1,-1,42,True
